# 新的第二部分（4.2.1）：Semi-log demand version

本 notebook 保留 $3\times4+1$ series-parallel network 與 optimization-based path-flow generation，只將 matching / time-to-value specification 改為 semi-log。

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from matplotlib import cm
from pathlib import Path

# ============================================================
# New Part 2 - Semi-log demand version
# Optimization-based path-flow generation on a general
# series-parallel network
#
# Default teacher example:
#   left_count  = 3
#   right_count = 4
#   num_direct  = 1
#
# Number of edges = left_count + right_count + num_direct = 8
# Number of paths = left_count * right_count + num_direct = 13
#
# Main difference from Part 1:
#   Part 1:
#       path price q_r directly generates path flow y_r.
#
#   Part 2:
#       path flow y is obtained by solving:
#
#       min_y sum_r [alpha_r - beta_r log(T_r(y) / T_REF) - q_r]^2
#       subject to y_r >= 0 and sum_r y_r = 1.
#
#   Then edge flow is computed by:
#       x = B y.
#
# Visualization:
#   This version plots ALL paths, not only the first six.
#   The path legend is placed outside the figure to avoid clutter.
# ============================================================


# ------------------------------------------------------------
# 0. Reproducibility
# ------------------------------------------------------------
np.random.seed(0)


# ------------------------------------------------------------
# 1. Network configuration
# ------------------------------------------------------------
left_count = 3
right_count = 4
num_direct = 1


# ------------------------------------------------------------
# 2. Learning parameters
# ------------------------------------------------------------
iterations = 30
I_0 = 4
v = 1.20
I_CAP = 80
noise_std = 0.05

# Semi-log reference travel time. If travel time is already normalized, use 1.0.
T_REF = 1.0


# ------------------------------------------------------------
# 3. Price bounds
# ------------------------------------------------------------
P_LOW = 0.05
P_HIGH = 50.0


# ------------------------------------------------------------
# 4. Output folder
# ------------------------------------------------------------
OUTPUT_DIR = Path("part2_series_parallel_semilog_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# Network construction
# ============================================================
def build_series_parallel_network(left_count=3, right_count=4, num_direct=1):
    """
    Build an L x R + D series-parallel network.

    Returns:
        edges      : list of edge names
        paths      : list of paths, each path is a list of edges
        path_names : readable path names
        B          : edge-path incidence matrix, shape = (E, A)
    """
    left_edges = [f"e{i+1}" for i in range(left_count)]
    right_edges = [f"f{j+1}" for j in range(right_count)]
    direct_edges = [f"g{k+1}" for k in range(num_direct)]

    edges = left_edges + right_edges + direct_edges
    paths = []
    path_names = []

    for e in left_edges:
        for f in right_edges:
            paths.append([e, f])
            path_names.append(f"{e}-{f}")

    for g in direct_edges:
        paths.append([g])
        path_names.append(g)

    B = np.zeros((len(edges), len(paths)))
    edge_index = {edge: idx for idx, edge in enumerate(edges)}

    for r, path in enumerate(paths):
        for edge in path:
            B[edge_index[edge], r] = 1.0

    return edges, paths, path_names, B


edges, paths, path_names, B = build_series_parallel_network(
    left_count=left_count,
    right_count=right_count,
    num_direct=num_direct
)

E = len(edges)
A = len(paths)

print("=" * 75)
print("Part 2: Optimization-based path-flow generation (Semi-log demand)")
print("=" * 75)
print(f"Network type        : {left_count} x {right_count} + {num_direct}")
print(f"Number of edges E   : {E}")
print(f"Number of paths A   : {A}")
print(f"Edges               : {edges}")
print(f"Paths               : {path_names}")
print(f"Incidence matrix B  : {B.shape}")
print("=" * 75)


# ============================================================
# Model parameters
# ============================================================

# Edge latency parameters: c_e(x_e) = a_e x_e + b_e
# Larger coefficients than Part 1 make congestion more visible.
a_e = np.linspace(3.00, 4.00, E)
b_e = np.linspace(0.10, 0.18, E)

# True path-level parameters used only to generate synthetic observations.
true_alpha = 3.00 + 0.04 * np.arange(A)
true_beta = 0.40 + 0.01 * np.arange(A)

# Initial estimated parameters.
alpha_hat = np.ones(A) * 3.0
beta_hat = np.ones(A) * 0.45

# Initial path prices q_r.
path_prices = np.linspace(0.20, 0.60, A)


# ============================================================
# Observation histories and simulation histories
# ============================================================
T_obs = [[] for _ in range(A)]
Lambda_obs = [[] for _ in range(A)]

alpha_hist = []
beta_hist = []
price_hist = []
path_flow_hist = []
edge_flow_hist = []
edge_latency_hist = []
path_latency_hist = []
gc_hist = []
flow_success_hist = []
iteration_logs = []


# ============================================================
# Core functions
# ============================================================
def compute_I_n(n):
    return int(min(I_CAP, max(I_0, I_0 * (v ** n))))


def compute_delta_n(I_n):
    return I_n ** (-1 / 4)


def compute_network_quantities(y):
    """
    Given path flow y:
        edge flow    x = B y
        edge latency c_e(x_e) = a_e x_e + b_e
        path latency T_r(y) = sum_e B_er c_e(x_e)
    """
    x = B @ y
    c = a_e * x + b_e
    T = B.T @ c
    return x, c, T


def log_latency(t_values, t_ref=T_REF):
    """Semi-log transform: log(T / T_REF). Requires strictly positive latency."""
    t_values = np.asarray(t_values, dtype=float)
    if np.any(t_values <= 0):
        raise ValueError("Semi-log demand requires strictly positive path latency values.")
    return np.log(t_values / t_ref)


def update_path_flows_optimization(path_prices, alpha, beta, y_prev=None, maxiter=100, ftol=1e-8):
    """
    Optimization-based path-flow generation.

    Solve:
        min_y sum_r [alpha_r - beta_r log(T_r(y) / T_REF) - q_r]^2
        s.t.  y_r >= 0, sum_r y_r = 1.

    The solution may be interior or near a simplex corner. A corner solution
    means most flow is assigned to one dominant path.
    """
    num_paths = B.shape[1]

    def objective(y):
        x, c, T = compute_network_quantities(y)
        lambda_hat = alpha - beta * log_latency(T)
        return float(np.sum((lambda_hat - path_prices) ** 2))

    constraints = ({
        "type": "eq",
        "fun": lambda y: np.sum(y) - 1.0
    },)

    bounds = [(0.0, 1.0) for _ in range(num_paths)]

    if y_prev is None:
        y0 = np.ones(num_paths) / num_paths
    else:
        y0 = np.clip(np.asarray(y_prev, dtype=float), 0.0, 1.0)
        if y0.sum() <= 0:
            y0 = np.ones(num_paths) / num_paths
        else:
            y0 = y0 / y0.sum()

    result = minimize(
        objective,
        y0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "ftol": float(ftol),
            "maxiter": int(maxiter),
            "disp": False,
        }
    )

    if result.success and np.isfinite(result.fun):
        y = result.x
        success = True
        message = str(result.message)
    else:
        y = y0
        success = False
        message = str(result.message)

    y = np.maximum(y, 0.0)
    y = y / y.sum() if y.sum() > 0 else np.ones(num_paths) / num_paths

    return y, success, message


def estimate_parameters(t_values, lambda_values, prev_alpha, prev_beta):
    """
    Estimate alpha and beta in:
        lambda = alpha - beta * log(T / T_REF)
    """
    t_values = np.asarray(t_values, dtype=float)
    lambda_values = np.asarray(lambda_values, dtype=float)

    z_values = log_latency(t_values)

    if len(z_values) < 3 or np.std(z_values) < 1e-8:
        return float(prev_alpha), float(prev_beta)

    def objective(params):
        alpha, beta = params
        pred = alpha - beta * z_values
        reg = 1e-4 * ((alpha - prev_alpha) ** 2 + (beta - prev_beta) ** 2)
        return float(np.sum((lambda_values - pred) ** 2) + reg)

    result = minimize(
        objective,
        x0=np.array([prev_alpha, prev_beta], dtype=float),
        method="L-BFGS-B",
        bounds=[(1e-6, None), (1e-6, None)]
    )

    if not result.success:
        return float(prev_alpha), float(prev_beta)

    return float(result.x[0]), float(result.x[1])


def solve_kkt_price(alpha_i, beta_i):
    """
    Bounded KKT-style price update.

    The unconstrained maximizer of:
        q_r * (alpha_r - beta_r q_r)
    is:
        q_r = alpha_r / (2 beta_r)

    Then the price is projected onto [P_LOW, P_HIGH].
    """
    beta_i = max(float(beta_i), 1e-8)
    q_unconstrained = float(alpha_i) / (2.0 * beta_i)
    return float(np.clip(q_unconstrained, P_LOW, P_HIGH))


def effective_number_of_paths(y):
    """
    Effective number of used paths:
        1 / sum_r y_r^2

    If all 13 paths are evenly used, this value is close to 13.
    If one path dominates, this value is close to 1.
    """
    return float(1.0 / np.sum(np.asarray(y) ** 2))


# ============================================================
# Learning loop
# ============================================================
y_prev = np.ones(A) / A

for n in range(iterations):
    I_n = compute_I_n(n)
    delta_n = compute_delta_n(I_n)
    q_hat = np.copy(path_prices)
    flow_success_all = True

    # A + 1 experimental intervals:
    #   m = 0       baseline prices
    #   m = 1,...,A perturb one path price at a time
    for m in range(A + 1):
        q_exp = np.copy(q_hat)

        if m > 0:
            perturbed_path = m - 1
            q_exp[perturbed_path] = q_hat[perturbed_path] + delta_n

        q_exp = np.clip(q_exp, P_LOW, P_HIGH)

        # Part 2: path flow is generated by solving the matching problem.
        y, success, msg = update_path_flows_optimization(
            path_prices=q_exp,
            alpha=alpha_hat,
            beta=beta_hat,
            y_prev=y_prev,
            maxiter=100,
            ftol=1e-8
        )

        flow_success_all = flow_success_all and success
        y_prev = y.copy()

        x, c, T = compute_network_quantities(y)

        # Generate I_n noisy observations under the same experimental condition.
        for _ in range(I_n):
            observed_lambda = (
                true_alpha
                - true_beta * log_latency(T)
                + np.random.normal(0, noise_std, size=A)
            )

            for r in range(A):
                T_obs[r].append(float(T[r]))
                Lambda_obs[r].append(float(observed_lambda[r]))

    # Estimate path-level alpha and beta.
    for r in range(A):
        alpha_hat[r], beta_hat[r] = estimate_parameters(
            T_obs[r],
            Lambda_obs[r],
            alpha_hat[r],
            beta_hat[r]
        )

    # Update path prices.
    for r in range(A):
        path_prices[r] = solve_kkt_price(alpha_hat[r], beta_hat[r])

    # Compute final stage quantities after price update.
    y, success, msg = update_path_flows_optimization(
        path_prices=path_prices,
        alpha=alpha_hat,
        beta=beta_hat,
        y_prev=y_prev,
        maxiter=100,
        ftol=1e-8
    )

    flow_success_all = flow_success_all and success
    y_prev = y.copy()

    x, c, T = compute_network_quantities(y)
    lambda_hat = alpha_hat - beta_hat * log_latency(T)
    generalized_loss = float(np.sum((lambda_hat - path_prices) ** 2))
    eff_paths = effective_number_of_paths(y)

    alpha_hist.append(alpha_hat.copy())
    beta_hist.append(beta_hat.copy())
    price_hist.append(path_prices.copy())
    path_flow_hist.append(y.copy())
    edge_flow_hist.append(x.copy())
    edge_latency_hist.append(c.copy())
    path_latency_hist.append(T.copy())
    gc_hist.append(generalized_loss)
    flow_success_hist.append(flow_success_all)

    iteration_logs.append({
        "Iteration": n + 1,
        "I_n": I_n,
        "FlowOK": flow_success_all,
        "AlphaMean": np.mean(alpha_hat),
        "BetaMean": np.mean(beta_hat),
        "PriceMin": np.min(path_prices),
        "PriceMax": np.max(path_prices),
        "FlowMin": np.min(y),
        "FlowMax": np.max(y),
        "EffectivePaths": eff_paths,
        "Loss": generalized_loss,
    })


# Convert histories to arrays.
alpha_hist = np.asarray(alpha_hist)
beta_hist = np.asarray(beta_hist)
price_hist = np.asarray(price_hist)
path_flow_hist = np.asarray(path_flow_hist)
edge_flow_hist = np.asarray(edge_flow_hist)
edge_latency_hist = np.asarray(edge_latency_hist)
path_latency_hist = np.asarray(path_latency_hist)
gc_hist = np.asarray(gc_hist)
flow_success_hist = np.asarray(flow_success_hist)


# ============================================================
# Compact log
# ============================================================
print("\nLearning log")
print("-" * 135)
for log in iteration_logs:
    print(
        f"Iter {log['Iteration']:2d} | "
        f"I_n={log['I_n']:3d} | "
        f"FlowOK={log['FlowOK']} | "
        f"mean(alpha)={log['AlphaMean']:.4f} | "
        f"mean(beta)={log['BetaMean']:.4f} | "
        f"price=[{log['PriceMin']:.3f}, {log['PriceMax']:.3f}] | "
        f"flow=[{log['FlowMin']:.4f}, {log['FlowMax']:.4f}] | "
        f"eff.paths={log['EffectivePaths']:.2f} | "
        f"loss={log['Loss']:.4f}"
    )


# ============================================================
# Visualization helpers
# ============================================================
def get_path_colors(num_paths):
    return cm.get_cmap("tab20", num_paths)


def style_axis(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.35)


def add_complete_path_legend(fig, colors, path_names, anchor=(0.89, 0.5)):
    """
    Add a complete path legend outside the figure.
    This is the key fix: all paths are displayed and all labels are shown
    without covering the panels.
    """
    handles = []
    labels = []

    for r, name in enumerate(path_names):
        h, = plt.plot([], [], color=colors(r), linewidth=1.6)
        handles.append(h)
        labels.append(f"Path {r+1}: {name}")

    fig.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=anchor,
        fontsize=8,
        ncol=1,
        frameon=True
    )


colors = get_path_colors(A)
iteration_axis = np.arange(1, iterations + 1)


# ============================================================
# Figure 1: Path-level learning dynamics
# All 13 paths are plotted. Legend is outside the figure.
# ============================================================
fig, axs = plt.subplots(2, 2, figsize=(15, 9))

for r in range(A):
    axs[0, 0].plot(iteration_axis, alpha_hist[:, r], color=colors(r), linewidth=1.4)
style_axis(
    axs[0, 0],
    "Estimated Alpha Evolution by Path (Semi-log)",
    "Iteration",
    r"Estimated $\hat{\alpha}_r$"
)

for r in range(A):
    axs[0, 1].plot(iteration_axis, beta_hist[:, r], color=colors(r), linewidth=1.4)
style_axis(
    axs[0, 1],
    "Estimated Beta Evolution by Path (Semi-log)",
    "Iteration",
    r"Estimated $\hat{\beta}_r$"
)

for r in range(A):
    axs[1, 0].plot(iteration_axis, price_hist[:, r], color=colors(r), linewidth=1.4)
style_axis(
    axs[1, 0],
    "Path Price Trajectories (Semi-log)",
    "Iteration",
    r"Path price $q_r$"
)

for r in range(A):
    axs[1, 1].plot(iteration_axis, path_flow_hist[:, r], color=colors(r), linewidth=1.4)
axs[1, 1].axhline(1.0 / A, color="black", linestyle="--", linewidth=1.0, alpha=0.75)
axs[1, 1].text(
    iterations * 0.55,
    1.0 / A,
    f"uniform benchmark = 1/{A}",
    fontsize=8,
    va="bottom"
)
style_axis(
    axs[1, 1],
    "Path Flow Trajectories (Semi-log)",
    "Iteration",
    r"Path flow $y_r$"
)

fig.suptitle(
    f"Part 2: Semi-log Optimization-Based Path-Flow Generation on a {left_count} x {right_count} + {num_direct} Network",
    fontsize=14,
    y=0.98
)

add_complete_path_legend(fig, colors, path_names, anchor=(0.89, 0.5))
plt.tight_layout(rect=[0, 0, 0.87, 0.95])

out1 = OUTPUT_DIR / "part2_semilog_all_paths_path_level.png"
plt.savefig(out1, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# Figure 2: Network-level diagnostics
# This figure demonstrates the network mapping x = B y.
# ============================================================
fig, axs = plt.subplots(2, 2, figsize=(15, 9))

for e in range(E):
    axs[0, 0].plot(iteration_axis, edge_flow_hist[:, e], linewidth=1.6, label=edges[e])
style_axis(
    axs[0, 0],
    r"Edge Flow Trajectories: $x = By$",
    "Iteration",
    r"Edge flow $x_e$"
)
axs[0, 0].legend(fontsize=8, ncol=2)

for e in range(E):
    axs[0, 1].plot(iteration_axis, edge_latency_hist[:, e], linewidth=1.6, label=edges[e])
style_axis(
    axs[0, 1],
    r"Edge Latency Trajectories: $c_e(x_e)$",
    "Iteration",
    r"Edge latency $c_e(x_e)$"
)
axs[0, 1].legend(fontsize=8, ncol=2)

for r in range(A):
    axs[1, 0].plot(iteration_axis, path_latency_hist[:, r], color=colors(r), linewidth=1.2)
style_axis(
    axs[1, 0],
    r"Path Latency Trajectories: $T_r(y)$",
    "Iteration",
    r"Path latency $T_r(y)$"
)

axs[1, 1].plot(iteration_axis, gc_hist, linewidth=2.0, label="matching loss")
axs[1, 1].set_title("Generalized Matching Loss and Effective Used Paths", fontsize=12)
axs[1, 1].set_xlabel("Iteration")
axs[1, 1].set_ylabel(r"$\sum_r(\hat{\lambda}_r - q_r)^2$")
axs[1, 1].grid(True, alpha=0.35)

ax2 = axs[1, 1].twinx()
effective_paths_hist = 1.0 / np.sum(path_flow_hist ** 2, axis=1)
ax2.plot(iteration_axis, effective_paths_hist, linestyle="--", linewidth=1.8, label="effective paths")
ax2.set_ylabel("Effective number of used paths")

lines_1, labels_1 = axs[1, 1].get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
axs[1, 1].legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=8, loc="best")

fig.suptitle(
    "Part 2 Semi-log Network-Level Diagnostics",
    fontsize=14,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

out2 = OUTPUT_DIR / "part2_semilog_network_diagnostics.png"
plt.savefig(out2, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# Figure 3: Separate high-resolution figures for thesis layout
# ============================================================
separate_figures = [
    ("part2_semilog_alpha_all_paths.png", alpha_hist, "Estimated Alpha Evolution by Path (Semi-log)", r"Estimated $\hat{\alpha}_r$"),
    ("part2_semilog_beta_all_paths.png", beta_hist, "Estimated Beta Evolution by Path (Semi-log)", r"Estimated $\hat{\beta}_r$"),
    ("part2_semilog_price_all_paths.png", price_hist, "Path Price Trajectories (Semi-log)", r"Path price $q_r$"),
    ("part2_semilog_flow_all_paths.png", path_flow_hist, "Path Flow Trajectories (Semi-log)", r"Path flow $y_r$"),
]

for filename, data, title, ylabel in separate_figures:
    fig, ax = plt.subplots(figsize=(11, 5.5))

    for r in range(A):
        ax.plot(iteration_axis, data[:, r], color=colors(r), linewidth=1.5, label=f"Path {r+1}: {path_names[r]}")

    if "Flow" in title:
        ax.axhline(1.0 / A, color="black", linestyle="--", linewidth=1.0, alpha=0.75)

    style_axis(ax, title, "Iteration", ylabel)

    ax.legend(
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        fontsize=8,
        ncol=1,
        frameon=True
    )

    plt.tight_layout()
    out = OUTPUT_DIR / filename
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# Final summary
# ============================================================
effective_paths_hist = 1.0 / np.sum(path_flow_hist ** 2, axis=1)

print("\nSaved figures to:", OUTPUT_DIR.resolve())
print("Main figures:")
print("  ", out1)
print("  ", out2)
print("\nFinal-stage summary")
print("-" * 75)
print(f"Final mean alpha             : {alpha_hist[-1].mean():.4f}")
print(f"Final mean beta              : {beta_hist[-1].mean():.4f}")
print(f"Final price range            : [{price_hist[-1].min():.4f}, {price_hist[-1].max():.4f}]")
print(f"Final path flow range        : [{path_flow_hist[-1].min():.4f}, {path_flow_hist[-1].max():.4f}]")
print(f"Final edge flow range        : [{edge_flow_hist[-1].min():.4f}, {edge_flow_hist[-1].max():.4f}]")
print(f"Final effective used paths   : {effective_paths_hist[-1]:.4f}")
print(f"Final generalized loss       : {gc_hist[-1]:.4f}")